<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/09_multimodal_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "your_email@example.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

src_path_str = str(SRC_DIR.resolve())
if src_path_str not in sys.path:
    sys.path.insert(0, src_path_str)

!pip install -q transformers pyarrow redivis

print("=" * 60)
print(f"Working Directory: {os.getcwd()}")
print(f"Module Search Path: {src_path_str}")
print("Repository synchronized and dependencies installed successfully.")
print("=" * 60)

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 108 (delta 40), reused 67 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 497.41 KiB | 1.77 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/DP-MMFL
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.4/230.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.4/820.4 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.3/408.3 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.1 MB/s eta 0:00:00
Working Directory: /content/DP-MMFL
Module Search Path: /content/DP-MMFL/src
Repository synchronized and dependencies installed

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
SRC_ROOT = PROJECT_ROOT / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chexpert_plus_manifest.parquet"
)

print("Project root:    ", PROJECT_ROOT)
print("Manifest:        ", MANIFEST_PATH)
print("Manifest exists: ", MANIFEST_PATH.exists())

Project root:     /content/drive/MyDrive/DP-MMFL
Manifest:         /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Manifest exists:  True


In [ ]:
import pandas as pd

manifest = pd.read_parquet(MANIFEST_PATH)
print("Manifest shape:  ", manifest.shape)
print("Columns:         ", len(manifest.columns))

Manifest shape:   (223462, 58)
Columns:          58


In [ ]:
required_columns = [
    "sample_id",
    "path_to_image",
    "deid_patient_id",
    "report_clean",
    "age",
    "sex",
    "race",
    "ethnicity",
    "experiment_split",
    "frontal_lateral",
    "ap_pa",
]

missing_columns = [
    col for col in required_columns
    if col not in manifest.columns
]

print("Missing required columns:", missing_columns)
assert not missing_columns, f"Missing required columns: {missing_columns}"
print("Required manifest columns: PASS")

Missing required columns: []
Required manifest columns: PASS


In [ ]:
from dp_mmfl.data.labels import TARGET_COLUMNS
from dp_mmfl.data.text import ClinicalTextTokenizer

print("Number of target labels:", len(TARGET_COLUMNS))
print("Targets:")
for i, label in enumerate(TARGET_COLUMNS, start=1):
    print(f"{i:2d}. {label}")

Number of target labels: 13
Targets:
 1. Enlarged Cardiomediastinum
 2. Cardiomegaly
 3. Lung Opacity
 4. Lung Lesion
 5. Edema
 6. Consolidation
 7. Pneumonia
 8. Atelectasis
 9. Pneumothorax
10. Pleural Effusion
11. Pleural Other
12. Fracture
13. Support Devices


In [ ]:
missing_targets = [
    f"target_{label}"
    for label in TARGET_COLUMNS
    if f"target_{label}" not in manifest.columns
]
missing_masks = [
    f"mask_{label}"
    for label in TARGET_COLUMNS
    if f"mask_{label}" not in manifest.columns
]

print("Missing target columns:", missing_targets)
print("Missing mask columns:  ", missing_masks)

assert not missing_targets, f"Missing targets: {missing_targets}"
assert not missing_masks, f"Missing masks: {missing_masks}"

print("13 target columns: PASS")
print("13 mask columns:   PASS")

Missing target columns: []
Missing mask columns:   []
13 target columns: PASS
13 mask columns:   PASS


In [ ]:
print("--- Full Projection Breakdown ---")
print(manifest["frontal_lateral"].value_counts(dropna=False))

frontal_manifest = manifest[
    manifest["frontal_lateral"] == "Frontal"
].copy()

print("\n--- Working Subset Profile ---")
print(f"Total images:        {len(manifest):,}")
print(f"Frontal images:      {len(frontal_manifest):,}")
print(f"Non-frontal images:  {len(manifest) - len(frontal_manifest):,}")

--- Full Projection Breakdown ---
frontal_lateral
Frontal    191071
Lateral     32391
Name: count, dtype: int64

--- Working Subset Profile ---
Total images:        223,462
Frontal images:      191,071
Non-frontal images:  32,391


In [ ]:
# Cell 9: Connect to Redivis Table
import redivis

# Connect to the CheXpert Plus PNG table descriptor
table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")
print("Redivis table:", table)

Redivis table: <Table aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj>


In [ ]:
# Cell 10: Fetch and Decode Single Frontal Image Stream
from io import BytesIO
from PIL import Image

# Extract first sample path
test_path = frontal_manifest.iloc[0]["path_to_image"]
print("CSV image path:")
print(test_path)

# Map CheXpert CSV path to Redivis table path
png_path = str(test_path)
if png_path.startswith("train/"):
    png_path = png_path[len("train/"):]
if png_path.endswith(".jpg"):
    png_path = png_path[:-4] + ".png"

print("\nRedivis PNG path:")
print(png_path)

# Read file directly into byte stream (in-memory, no disk save)
file = table.file(png_path)
data = file.read(as_text=False)
print("\nDownloaded bytes:", len(data))

# Decode byte stream
image = Image.open(BytesIO(data))
print("Format:", image.format)
print("Mode:  ", image.mode)
print("Size:  ", image.size)

# Force load into memory to verify decoding integrity
image.load()
print("\nImage decode: PASS")

CSV image path:
train/patient00003/study1/view1_frontal.jpg

Redivis PNG path:
patient00003/study1/view1_frontal.png
Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=c459e3dd88e5e46a4d033d452f784d6d

Downloaded bytes: 3163254
Format: PNG
Mode:   L
Size:   (2828, 2320)

Image decode: PASS


In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/data/dataset.py
from io import BytesIO
from pathlib import Path
from typing import Optional, Union
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

from dp_mmfl.data.labels import TARGET_COLUMNS
from dp_mmfl.data.text import ClinicalTextTokenizer


class CheXpertPlusMultimodalDataset(Dataset):
    """
    Multimodal PyTorch Dataset for CheXpert Plus.
    Streams frontal chest X-rays on-demand from Redivis and tokenizes clinical reports.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        table,
        image_transform=None,
        tokenizer: Optional[ClinicalTextTokenizer] = None,
        split: Optional[str] = None,
    ):
        self.manifest = pd.read_parquet(manifest_path)

        # Filter for frontal projections only
        self.manifest = self.manifest[
            self.manifest["frontal_lateral"] == "Frontal"
        ].copy()

        # Optional filter by experiment split
        if split is not None:
            self.manifest = self.manifest[
                self.manifest["experiment_split"] == split
            ].copy()

        self.manifest = self.manifest.reset_index(drop=True)
        self.table = table
        self.image_transform = image_transform
        self.tokenizer = tokenizer or ClinicalTextTokenizer()

    def __len__(self) -> int:
        return len(self.manifest)

    def _map_to_redivis_path(self, path_to_image: str) -> str:
        png_path = str(path_to_image)
        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]
        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"
        return png_path

    def __getitem__(self, idx: int):
        row = self.manifest.iloc[idx]

        # 1. Fetch and process image
        png_path = self._map_to_redivis_path(row["path_to_image"])
        file = self.table.file(png_path)
        data = file.read(as_text=False)
        image = Image.open(BytesIO(data)).convert("L")

        if self.image_transform is not None:
            image_tensor = self.image_transform(image)
        else:
            image_tensor = image

        # 2. Tokenize report_clean
        text_encoded = self.tokenizer.encode(str(row["report_clean"]))
        input_ids = text_encoded["input_ids"].squeeze(0)
        attention_mask = text_encoded["attention_mask"].squeeze(0)

        # 3. Extract 13 multi-label targets and U-Ignore masks
        target_cols = [f"target_{col}" for col in TARGET_COLUMNS]
        mask_cols = [f"mask_{col}" for col in TARGET_COLUMNS]

        labels = torch.tensor(row[target_cols].to_numpy(dtype=float), dtype=torch.float32)
        label_mask = torch.tensor(row[mask_cols].to_numpy(dtype=float), dtype=torch.float32)

        # 4. Extract patient metadata
        metadata = {
            "sample_id": row["sample_id"],
            "patient_id": row["deid_patient_id"],
            "age": row["age"],
            "sex": row["sex"],
            "race": row["race"],
            "ethnicity": row["ethnicity"],
            "experiment_split": row["experiment_split"],
            "frontal_lateral": row["frontal_lateral"],
            "ap_pa": row["ap_pa"],
        }

        return {
            "image": image_tensor,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "label_mask": label_mask,
            "metadata": metadata,
        }

Writing /content/DP-MMFL/src/dp_mmfl/data/dataset.py


In [ ]:
%cd /content/DP-MMFL
!git add src/dp_mmfl/data/dataset.py
!git commit -m "implemented CheXpertPlusMultimodalDataset"
!git push origin main

/content/DP-MMFL
[main a437ad3] implemented CheXpertPlusMultimodalDataset
 1 file changed, 102 insertions(+)
 create mode 100644 src/dp_mmfl/data/dataset.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 1.60 KiB | 1.60 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   015ad1b..a437ad3  main -> main


In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224
RESIZE_SIZE = 256

def get_train_transforms():
    return transforms.Compose([
        transforms.Resize(RESIZE_SIZE, antialias=True),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

def get_eval_transforms():
    return transforms.Compose([
        transforms.Resize(RESIZE_SIZE, antialias=True),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

print("Transforms defined successfully.")

Transforms defined successfully.


In [ ]:
from dp_mmfl.data.text import ClinicalTextTokenizer

text_tokenizer = ClinicalTextTokenizer()
print("Tokenizer: ", text_tokenizer.model_name)
print("Max length:", text_tokenizer.max_length)
print("Vocab size:", text_tokenizer.tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Tokenizer:  emilyalsentzer/Bio_ClinicalBERT
Max length: 384
Vocab size: 28996


In [ ]:
from dp_mmfl.data.dataset import CheXpertPlusMultimodalDataset

eval_transform = get_eval_transforms()

dataset = CheXpertPlusMultimodalDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    image_transform=eval_transform,
    tokenizer=text_tokenizer,
)

print("Dataset size:", len(dataset))
assert len(dataset) == 191071, f"Expected 191,071 samples, found {len(dataset)}"

Dataset size: 191071


In [ ]:
import torch

sample = dataset[0]

print("Sample keys:")
print(sample.keys())

print("\n--- Image ---")
print("Shape:", sample["image"].shape)
print("Dtype:", sample["image"].dtype)

print("\n--- Text ---")
print("input_ids shape:     ", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("input_ids dtype:     ", sample["input_ids"].dtype)
print("attention_mask dtype:", sample["attention_mask"].dtype)

print("\n--- Labels ---")
print("Shape: ", sample["labels"].shape)
print("Values:", sample["labels"])

print("\n--- Label Mask ---")
print("Shape: ", sample["label_mask"].shape)
print("Values:", sample["label_mask"])

print("\n--- Metadata ---")
for key, value in sample["metadata"].items():
    print(f"{key:16s}: {value}")

# Structural assertions
assert sample["image"].shape == torch.Size([3, 224, 224]), "Image shape mismatch!"
assert sample["image"].dtype == torch.float32, "Image dtype mismatch!"
assert sample["input_ids"].shape == torch.Size([384]), "input_ids length mismatch!"
assert sample["attention_mask"].shape == torch.Size([384]), "attention_mask length mismatch!"
assert sample["input_ids"].dtype == torch.int64, "input_ids dtype mismatch!"
assert sample["attention_mask"].dtype == torch.int64, "attention_mask dtype mismatch!"
assert sample["labels"].shape == torch.Size([13]), "Labels vector size mismatch!"
assert sample["label_mask"].shape == torch.Size([13]), "Label mask vector size mismatch!"
assert sample["labels"].dtype == torch.float32, "Labels dtype mismatch!"
assert sample["label_mask"].dtype == torch.float32, "Label mask dtype mismatch!"

print("\nMultimodal sample verification: ALL ASSERTIONS PASSED")

Sample keys:
dict_keys(['image', 'input_ids', 'attention_mask', 'labels', 'label_mask', 'metadata'])

--- Image ---
Shape: torch.Size([3, 224, 224])
Dtype: torch.float32

--- Text ---
input_ids shape:      torch.Size([384])
attention_mask shape: torch.Size([384])
input_ids dtype:      torch.int64
attention_mask dtype: torch.int64

--- Labels ---
Shape:  torch.Size([13])
Values: tensor([nan, nan, 1., nan, nan, nan, nan, nan, 0., nan, nan, nan, nan])

--- Label Mask ---
Shape:  torch.Size([13])
Values: tensor([0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])

--- Metadata ---
sample_id       : 0
patient_id      : patient00003
age             : 41.0
sex             : Male
race            : White
ethnicity       : Non-Hispanic/Non-Latino
experiment_split: train
frontal_lateral : Frontal
ap_pa           : AP

Multimodal sample verification: ALL ASSERTIONS PASSED


In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/data/dataset.py
from io import BytesIO
from pathlib import Path
from typing import Optional, Union
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

from dp_mmfl.data.labels import TARGET_COLUMNS
from dp_mmfl.data.text import ClinicalTextTokenizer


class CheXpertPlusMultimodalDataset(Dataset):
    """
    Multimodal PyTorch Dataset for CheXpert Plus.
    Streams frontal chest X-rays on-demand from Redivis and tokenizes clinical reports.
    Converts unmentioned/NaN targets to 0.0 with mask=0.0 under the U-Ignore policy.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        table,
        image_transform=None,
        tokenizer: Optional[ClinicalTextTokenizer] = None,
        split: Optional[str] = None,
    ):
        self.manifest = pd.read_parquet(manifest_path)

        # Filter for frontal projections only
        self.manifest = self.manifest[
            self.manifest["frontal_lateral"] == "Frontal"
        ].copy()

        # Optional filter by experiment split
        if split is not None:
            self.manifest = self.manifest[
                self.manifest["experiment_split"] == split
            ].copy()

        self.manifest = self.manifest.reset_index(drop=True)
        self.table = table
        self.image_transform = image_transform
        self.tokenizer = tokenizer or ClinicalTextTokenizer()

    def __len__(self) -> int:
        return len(self.manifest)

    def _map_to_redivis_path(self, path_to_image: str) -> str:
        png_path = str(path_to_image)
        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]
        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"
        return png_path

    def __getitem__(self, idx: int):
        row = self.manifest.iloc[idx]

        # 1. Fetch and process image
        png_path = self._map_to_redivis_path(row["path_to_image"])
        file = self.table.file(png_path)
        data = file.read(as_text=False)
        image = Image.open(BytesIO(data)).convert("L")

        if self.image_transform is not None:
            image_tensor = self.image_transform(image)
        else:
            image_tensor = image

        # 2. Tokenize report_clean
        text_encoded = self.tokenizer.encode(str(row["report_clean"]))
        input_ids = text_encoded["input_ids"].squeeze(0)
        attention_mask = text_encoded["attention_mask"].squeeze(0)

        # 3. Target labels and U-Ignore masks (strict NaN -> 0.0 conversion)
        labels = torch.tensor(
            [
                0.0 if pd.isna(row[f"target_{label}"])
                else float(row[f"target_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )
        label_mask = torch.tensor(
            [
                float(row[f"mask_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        # 4. Patient metadata
        metadata = {
            "sample_id": row["sample_id"],
            "patient_id": row["deid_patient_id"],
            "age": row["age"],
            "sex": row["sex"],
            "race": row["race"],
            "ethnicity": row["ethnicity"],
            "experiment_split": row["experiment_split"],
            "frontal_lateral": row["frontal_lateral"],
            "ap_pa": row["ap_pa"],
        }

        return {
            "image": image_tensor,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "label_mask": label_mask,
            "metadata": metadata,
        }

Overwriting /content/DP-MMFL/src/dp_mmfl/data/dataset.py


In [ ]:
%cd /content/DP-MMFL
!git add src/dp_mmfl/data/dataset.py
!git commit -m "fix(dataset): sanitize NaN targets to 0.0 via explicit row-level check"
!git push origin main

/content/DP-MMFL
[main 885c7d0] fix(dataset): sanitize NaN targets to 0.0 via explicit row-level check
 1 file changed, 20 insertions(+), 9 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 769 bytes | 769.00 KiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   a437ad3..885c7d0  main -> main


In [ ]:
import importlib
import torch
import pandas as pd
import dp_mmfl.data.dataset as dataset_module

importlib.reload(dataset_module)
CheXpertPlusMultimodalDataset = dataset_module.CheXpertPlusMultimodalDataset

dataset = CheXpertPlusMultimodalDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    image_transform=eval_transform,
    tokenizer=text_tokenizer,
)

sample = dataset[0]
print("Labels:    ", sample["labels"])
print("Label Mask:", sample["label_mask"])

# Structural and value assertions
assert sample["labels"].shape == torch.Size([13]), "Labels shape mismatch!"
assert sample["label_mask"].shape == torch.Size([13]), "Label mask shape mismatch!"
assert not torch.isnan(sample["labels"]).any(), "NaN found in sanitized labels vector!"
assert not torch.isnan(sample["label_mask"]).any(), "NaN found in label mask vector!"
assert torch.all(
    (sample["labels"] == 0.0) | (sample["labels"] == 1.0)
), "Non-binary values present in labels vector!"
assert torch.all(
    (sample["label_mask"] == 0.0) | (sample["label_mask"] == 1.0)
), "Non-binary values present in label mask vector!"

print("\nU-Ignore label representation: PASS")

Labels:     tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Label Mask: tensor([0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])

U-Ignore label representation: PASS


In [ ]:

from torch.utils.data import DataLoader

In [ ]:
#Create Validation DataLoader (Single-worker)
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
)
print("DataLoader created successfully.")

DataLoader created successfully.


In [ ]:
#Retrieve One Batch and Validate Shapes
batch = next(iter(loader))
print("Batch retrieved successfully.")
print("\nKeys:")
print(batch.keys())

print("\n--- Tensor Shapes ---")
print("image:         ", batch["image"].shape)
print("input_ids:     ", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:        ", batch["labels"].shape)
print("label_mask:    ", batch["label_mask"].shape)

Batch retrieved successfully.

Keys:
dict_keys(['image', 'input_ids', 'attention_mask', 'labels', 'label_mask', 'metadata'])

--- Tensor Shapes ---
image:          torch.Size([4, 3, 224, 224])
input_ids:      torch.Size([4, 384])
attention_mask: torch.Size([4, 384])
labels:         torch.Size([4, 13])
label_mask:     torch.Size([4, 13])


In [ ]:
# Cell 18: Verify Batch Data Types
print("\n--- Tensor Dtypes ---")
print("image:         ", batch["image"].dtype)
print("input_ids:     ", batch["input_ids"].dtype)
print("attention_mask:", batch["attention_mask"].dtype)
print("labels:        ", batch["labels"].dtype)
print("label_mask:    ", batch["label_mask"].dtype)


--- Tensor Dtypes ---
image:          torch.float32
input_ids:      torch.int64
attention_mask: torch.int64
labels:         torch.float32
label_mask:     torch.float32


In [ ]:
# Cell 19: Validate Batch Structural and Numerical Integrity
assert batch["image"].shape == (4, 3, 224, 224), f"Unexpected image shape: {batch['image'].shape}"
assert batch["input_ids"].shape == (4, 384), f"Unexpected input_ids shape: {batch['input_ids'].shape}"
assert batch["attention_mask"].shape == (4, 384), f"Unexpected attention_mask shape: {batch['attention_mask'].shape}"
assert batch["labels"].shape == (4, 13), f"Unexpected labels shape: {batch['labels'].shape}"
assert batch["label_mask"].shape == (4, 13), f"Unexpected label_mask shape: {batch['label_mask'].shape}"

assert batch["image"].dtype == torch.float32, f"Unexpected image dtype: {batch['image'].dtype}"
assert batch["input_ids"].dtype == torch.int64, f"Unexpected input_ids dtype: {batch['input_ids'].dtype}"
assert batch["attention_mask"].dtype == torch.int64, f"Unexpected attention_mask dtype: {batch['attention_mask'].dtype}"
assert batch["labels"].dtype == torch.float32, f"Unexpected labels dtype: {batch['labels'].dtype}"
assert batch["label_mask"].dtype == torch.float32, f"Unexpected label_mask dtype: {batch['label_mask'].dtype}"

assert not torch.isnan(batch["labels"]).any(), "NaN detected in batched labels tensor!"
assert not torch.isnan(batch["label_mask"]).any(), "NaN detected in batched label_mask tensor!"

assert torch.all(
    (batch["labels"] == 0.0) | (batch["labels"] == 1.0)
), "Non-binary targets found in labels tensor!"
assert torch.all(
    (batch["label_mask"] == 0.0) | (batch["label_mask"] == 1.0)
), "Non-binary masks found in label_mask tensor!"

print("DataLoader batch validation: ALL CHECKS PASSED")

DataLoader batch validation: ALL CHECKS PASSED


In [ ]:
# Cell 20: Inspect Collation of Metadata Dictionary
print("\n--- Metadata Collation Inspection ---")
for key, value in batch["metadata"].items():
    print(f"\n{key} ({type(value)}):")
    print(value)


--- Metadata Collation Inspection ---

sample_id (<class 'torch.Tensor'>):
tensor([0, 4, 6, 7])

patient_id (<class 'list'>):
['patient00003', 'patient00009', 'patient00016', 'patient00021']

age (<class 'torch.Tensor'>):
tensor([41., 76., 54., 36.], dtype=torch.float64)

sex (<class 'list'>):
['Male', 'Male', 'Female', 'Female']

race (<class 'list'>):
['White', 'Asian', 'Asian', 'Asian']

ethnicity (<class 'list'>):
['Non-Hispanic/Non-Latino', 'Non-Hispanic/Non-Latino', 'Non-Hispanic/Non-Latino', 'Non-Hispanic/Non-Latino']

experiment_split (<class 'list'>):
['train', 'train', 'train', 'train']

frontal_lateral (<class 'list'>):
['Frontal', 'Frontal', 'Frontal', 'Frontal']

ap_pa (<class 'list'>):
['AP', 'PA', 'PA', 'PA']


In [ ]:
#Device Identification
import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM Allocated:   ", f"{torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")
    print("VRAM Reserved:    ", f"{torch.cuda.memory_reserved(0) / (1024**2):.2f} MB")

Device: cuda:0
GPU: Tesla T4
VRAM Allocated:    2.32 MB
VRAM Reserved:     22.00 MB


In [ ]:
#Transfer Tensors to Target Device
image = batch["image"].to(device)
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)
label_mask = batch["label_mask"].to(device)

print("--- Device Placement ---")
print("image:         ", image.device)
print("input_ids:     ", input_ids.device)
print("attention_mask:", attention_mask.device)
print("labels:        ", labels.device)
print("label_mask:    ", label_mask.device)

--- Device Placement ---
image:          cuda:0
input_ids:      cuda:0
attention_mask: cuda:0
labels:         cuda:0
label_mask:     cuda:0


In [ ]:
#Final Device and Dimensional Integrity Assertions
expected_type = device.type

assert image.device.type == expected_type, f"image device mismatch: {image.device}"
assert input_ids.device.type == expected_type, f"input_ids device mismatch: {input_ids.device}"
assert attention_mask.device.type == expected_type, f"attention_mask device mismatch: {attention_mask.device}"
assert labels.device.type == expected_type, f"labels device mismatch: {labels.device}"
assert label_mask.device.type == expected_type, f"label_mask device mismatch: {label_mask.device}"

assert image.shape == (4, 3, 224, 224), f"image shape mismatch: {image.shape}"
assert input_ids.shape == (4, 384), f"input_ids shape mismatch: {input_ids.shape}"
assert attention_mask.shape == (4, 384), f"attention_mask shape mismatch: {attention_mask.shape}"
assert labels.shape == (4, 13), f"labels shape mismatch: {labels.shape}"
assert label_mask.shape == (4, 13), f"label_mask shape mismatch: {label_mask.shape}"

assert not torch.isnan(labels).any(), "NaN found in GPU-bound labels tensor!"
assert not torch.isnan(label_mask).any(), "NaN found in GPU-bound label_mask tensor!"

print("GPU batch transfer validation: ALL CHECKS PASSED")

GPU batch transfer validation: ALL CHECKS PASSED
